In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

os.chdir('..')
assert os.path.basename(os.getcwd()) == "munich-accident-analysis", \
    "Need to be in the repo directory, restart the notebook and run this cell again"
os.makedirs("data/plots", exist_ok=True)

# D_Beteiligte and D_Mitfahrer data analysis
- Please run the 'preprocessing/convert_data.r' script to get the csv files before running this notebook.
- The .RData source file used in 'preprocessing/convert_data.r' script to get the csv files!
- The meaning of the columns / findings / questions are written in a document in drive.

## Load the tables

In [ ]:
# Data Paths
beteiligte_table_path = "data/D_Beteiligte_RData.csv"
mitfahrer_table_path = "data/D_Mitfahrer_RData.csv"
# For merging the with the tables above
unfall_table_path = "data/D_Unfall_RData.csv"

In [ ]:
beteiligte_table = pd.read_csv(beteiligte_table_path)
mitfahrer_table = pd.read_csv(mitfahrer_table_path)
unfall_table = pd.read_csv(unfall_table_path)

In [ ]:
print("Shape: ", beteiligte_table.shape)
beteiligte_table.head(3)

In [ ]:
print("Shape: ", mitfahrer_table.shape)
mitfahrer_table.head(3)

In [ ]:
print("Shape: ", unfall_table.shape)
unfall_table.head(3)

## Beteiligte Table Analysis

In [ ]:
# Merge and preprocess the tables
unfall_beteiligte_merged = pd.merge(beteiligte_table, unfall_table[['UN_KEY', 'DateTime', 'Unfalltyp']], on="UN_KEY")
unfall_beteiligte_merged["DateTime"] = pd.to_datetime(unfall_beteiligte_merged["DateTime"], errors="coerce")
unfall_beteiligte_merged["Year"] = unfall_beteiligte_merged["DateTime"].dt.year
unfall_beteiligte_merged["Month"] = unfall_beteiligte_merged["DateTime"].dt.month
unfall_beteiligte_merged["Year_Month"] = unfall_beteiligte_merged["DateTime"].dt.to_period("M")

unfall_beteiligte_merged.shape

In [ ]:
unfall_beteiligte_merged.head(3)

In [ ]:
unfall_beteiligte_merged.info()

In [ ]:
# Check one example accident
accident_id = "00000_0001095383"
unfall_beteiligte_merged[unfall_beteiligte_merged["UN_KEY"] == accident_id]

In [ ]:
unfall_beteiligte_merged.UN_KEY.nunique()

In [ ]:
unfall_beteiligte_merged.describe()

In [ ]:
# Check the unique values of the columns except for the UN_KEY and BTA_FDATUM columns
for col in unfall_beteiligte_merged.columns:
    if col not in ["UN_KEY", "BTA_FDATUM"]:
        print(f"Column: {col}")
        print(unfall_beteiligte_merged[col].unique())
        print("\n")

In [ ]:
# HISTOGRAMS for some columns

# Create a 2x2 grid of subplots
fig, axes = plt.subplots(4, 3, figsize=(15, 15))  # Adjust size as needed

# Plot the histograms in each subplot
sns.histplot(unfall_beteiligte_merged.UN_KEY.value_counts(), bins=12, color="blue", ax=axes[0, 0], stat="probability")
axes[0, 0].set_title("Distr of number of parties involved in accidents")

sns.countplot(x=unfall_beteiligte_merged.BT_SEX, color="blue", ax=axes[0, 1], stat="probability")
axes[0, 1].set_title("Sex Distr.")

sns.histplot(unfall_beteiligte_merged.BT_ALTER, bins=10, color="blue", ax=axes[0, 2], stat="probability")
axes[0, 2].set_title("Age Distr.")

sns.histplot(unfall_beteiligte_merged.BT_FLUCHT, bins=2, color="brown", ax=axes[1, 0], stat="probability")
axes[1, 0].set_title("Hit and Run Distr.")

sns.histplot(unfall_beteiligte_merged.BT_FOLGEN, bins=3, color="brown", ax=axes[1, 1], stat="probability")
axes[1, 1].set_title("Consequences Distr. (1: Death, 2: Severe Injury, 3: Light Injury")

sns.histplot(unfall_beteiligte_merged.BT_BETEIL, color="brown", ax=axes[1, 2], stat="probability")
axes[1, 2].set_title("Type of Traffic Involvement Distr. ? ")

sns.histplot(unfall_beteiligte_merged.BTA_AUSW, bins=2, color="red", ax=axes[2, 0], stat="probability")
axes[2, 0].set_title("Driver License Availability Distr.")

sns.histplot(unfall_beteiligte_merged.BT_ALKOH, bins=2, color="red", ax=axes[2, 1], stat="probability")
axes[2, 1].set_title("Alcohol Consumption Distr.")

sns.histplot(unfall_beteiligte_merged.BT_DROGEN, bins=2, color="red", ax=axes[2, 2], stat="probability")
axes[2, 2].set_title("Drug Consumption Distr.")

sns.histplot(unfall_beteiligte_merged.BT_BETEIL_K, color="orange", ax=axes[3, 0], stat="probability")
axes[3, 0].set_title("Vehicle Type Distr.")
axes[3, 0].tick_params(axis='x', rotation=90)

sns.histplot(unfall_beteiligte_merged.Unfallursache_1, color="orange", ax=axes[3, 1], stat="probability")
axes[3, 1].set_title("Accident Cause Distr.")

sns.histplot(unfall_beteiligte_merged.BTA_FALTER, color="orange", ax=axes[3, 2], stat="probability")
axes[3, 2].set_title("Driving Licence Age Distr.")

# Adjust layout to prevent overlap
plt.tight_layout()
# Show the plots
plt.show()

In [ ]:
# Hit and run accidents ratio
unfall_beteiligte_merged.BT_FLUCHT.value_counts(normalize=True)

In [ ]:
# Accident Consequent distrb
unfall_beteiligte_merged.BT_FOLGEN.value_counts(normalize=True)

In [ ]:
# Map numeric values to the corresponding labels
bt_folgen_labels = {1: "Deceased", 2: "Seriously injured", 3: "Lightly injured"}

# Group the data by Year and BT_FOLGEN
conseq_yearly = unfall_beteiligte_merged.groupby(["Year", "BT_FOLGEN"]).size().unstack()

# Create a figure with 1 row and 3 columns
fig, axes = plt.subplots(1, 3, figsize=(18, 6))  # 1 row, 3 columns

# Plot each BT_FOLGEN value in a separate subplot
for i, bt_folgen in enumerate(conseq_yearly.columns):
    axes[i].plot(conseq_yearly.index, conseq_yearly[bt_folgen], marker='o')
    axes[i].set_title(f"{bt_folgen_labels[bt_folgen]} over years")
    axes[i].set_xlabel("Year")
    axes[i].set_ylabel("Number of Accidents")
    axes[i].grid(True)
    axes[i].legend([bt_folgen_labels[bt_folgen]])

# Adjust layout to avoid overlap
plt.tight_layout()
plt.show()

In [ ]:
# Group the data by Year-Month and BT_FOLGEN
conseq_monthly = unfall_beteiligte_merged.groupby(["Year_Month", "BT_FOLGEN"]).size().unstack()

# Create a figure with 1 row and 3 columns for each BT_FOLGEN category
fig, axes = plt.subplots(1, 3, figsize=(18, 6))  # 1 row, 3 columns

# Plot each BT_FOLGEN value in a separate subplot
for i, bt_folgen in enumerate(conseq_monthly.columns):
    axes[i].plot(conseq_monthly.index.astype(str), conseq_monthly[bt_folgen], marker='o')
    axes[i].set_title(f"Accident Consequences for {bt_folgen_labels[bt_folgen]} over the months")
    axes[i].set_xlabel("Month/Year")
    axes[i].grid(True)
    axes[i].legend([bt_folgen_labels[bt_folgen]])

    # Customize x-axis ticks to show only 1, 6, and 12 for each year
    tick_labels = [str(period) for period in conseq_monthly.index if period.month in [1, 6]]
    axes[i].set_xticks(tick_labels)
    axes[i].set_xticklabels(tick_labels)
    # Tilting the x-axis labels
    axes[i].tick_params(axis='x', rotation=45)

# Adjust layout to avoid overlap
plt.tight_layout()
plt.show()

In [ ]:
# Consequences by unfalltyp
unfalltyp_consequence = unfall_beteiligte_merged.groupby(["Unfalltyp", "BT_FOLGEN"]).size().unstack()
unfalltyp_consequence

In [ ]:
# Unfallursache_1
unfall_beteiligte_merged.Unfallursache_1.unique()

In [ ]:
cross_table = pd.crosstab(unfall_beteiligte_merged['Unfallursache_1'], unfall_beteiligte_merged['Unfalltyp'])
cross_table

In [ ]:
cross_table = pd.crosstab(unfall_beteiligte_merged['Unfallursache_1'], unfall_beteiligte_merged['Unfalltyp'], normalize='index') * 100
cross_table

In [ ]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(cross_table)
print("Chi-squared test statistic:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)

In [ ]:
cross_table.plot(kind='bar', stacked=True, figsize=(10, 6))
plt.xlabel('Unfallursache_1 (Accident Cause)')
plt.ylabel('Count')
plt.title('Stacked Bar Chart of Unfalltyp (Accident Type) by Unfallursache_1 (Accident Cause)')
plt.legend(title='Unfalltyp')
plt.show()

In [ ]:
# Plot the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cross_table, annot=True, fmt=".1f", cmap="YlGnBu")
plt.title("Heatmap of Unfalltyp by Unfallursache_1 (Row-Normalized Percentages)")
plt.xlabel("Unfalltyp (Accident Type)")
plt.ylabel("Unfallursache_1 (Accident Cause)")
plt.show()

## Mitfahrer Table

In [ ]:
mitfahrer_table.head(3)

In [ ]:
# Only in 3K accidents, mitfahrer is logged
mitfahrer_table.UN_KEY.nunique()

In [ ]:
# Check the unique values of the columns except for the UN_KEY and BTA_FDATUM columns
for col in mitfahrer_table.columns:
    if col not in ["UN_KEY"]:
        print(f"Column: {col}")
        print(mitfahrer_table[col].unique())
        print("\n")

In [ ]:
# HISTOGRAMS for some columns

# Create a 2x2 grid of subplots
fig, axes = plt.subplots(2, 3, figsize=(15, 15))  # Adjust size as needed

# Plot the histograms in each subplot
sns.histplot(mitfahrer_table.UN_KEY.value_counts(), bins=12, color="blue", ax=axes[0, 0])
axes[0, 0].set_title("Distr of number of passengers in accidents")

sns.histplot(mitfahrer_table.MI_ALTER, bins=10, color="blue", ax=axes[0, 1])
axes[0, 1].set_title("Passenger Age Distr.")

sns.histplot(mitfahrer_table.MI_SEX, bins=2, color="red", ax=axes[0, 2])
axes[0, 2].set_title("Passenger Sex Distr. ?")

sns.histplot(mitfahrer_table.MI_FOLGEN, bins=3, color="red", ax=axes[1, 0])
axes[1, 0].set_title("Folgen Distr. ???")

sns.histplot(mitfahrer_table.BT_BETEIL_K, color="orange", ax=axes[1, 1])
axes[1, 1].set_title("Vehicle Type Distr. of Passengers")
axes[1, 1].tick_params(axis='x', rotation=90)

# Adjust layout to prevent overlap
plt.tight_layout()
# Show the plots
plt.show()

In [ ]:
# Consequences
mitfahrer_table.MI_FOLGEN.value_counts(normalize=True)

In [ ]:
cross_tab = pd.crosstab(unfall_beteiligte_merged['Unfalltyp'], unfall_beteiligte_merged['BT_BETEIL_K'], normalize='index')

In [ ]:
unfall_beteiligte_merged.head()

## Analyze vehicle distribution among accident types

Since in in the table for Beteiligte are more than one observation per accident, since also sometimes multiple parties are involved we need to account for multiple different vehicles being present in a single accident. Therefore we create a column for each vehicle class and the value in there indicates the amount of this vehicle in this setting. E.g. if two parties are involved, one bike and one car, we want 1 in the car column and 1 in the bike columns, 0 in all other.

In [ ]:
temp_df = beteiligte_table.copy()
categories = list(temp_df.BT_BETEIL_K.unique())
for cat in categories:
    temp_df[f'BT_BETEIL_K_{cat}'] = (temp_df['BT_BETEIL_K'] == cat).astype(int)
beteiligte_features_vehicle_type_counts = temp_df.groupby(
    'UN_KEY', as_index=False
)[[f'BT_BETEIL_K_{cat}' for cat in categories]].sum()
beteiligte_features_vehicle_type_counts.rename(
    columns={f'BT_BETEIL_K_{cat}': f'numerical_BT_BETEIL_K_{cat}' for cat in categories},
    inplace=True
)
beteiligte_features_vehicle_type_counts = beteiligte_features_vehicle_type_counts.merge(unfall_table[['UN_KEY', 'Unfalltyp']], on="UN_KEY",how="left")

In [ ]:
grouped_vehicle_counts = beteiligte_features_vehicle_type_counts.groupby('Unfalltyp')[beteiligte_features_vehicle_type_counts.columns[1:-1]].sum()
row_totals = grouped_vehicle_counts.sum(axis=1)
proportions = grouped_vehicle_counts.div(row_totals, axis=0)

In [ ]:
beteiligte_features_vehicle_type_counts

In [ ]:
vehicle_types = ["Car", "Truck", "Unknown (Hit and run)", "Pedestrian", "Bike", "Other", "Tram", "Bus", "Motorcycle", "E-Scooter"]
accident_types = ["1: Driving accident", "2: Turning accident", "3: Entering/Crossing accident", "4: Crossing accident (pedestrian)", "5: Accident caused by stationary traffic", "6: Accident in longitudinal traffic", "7: Other accident"]
ax = proportions.plot(kind='bar', stacked=True, colormap="Paired", figsize=(7,6))
ax.legend_.remove()
plt.legend(vehicle_types, title="Vehicle Type", loc="upper left", bbox_to_anchor=(1.05, 1))
ax.set_xticks(range(7))
ax.set_xticklabels(accident_types, rotation=45, ha='right')

plt.xlabel('Accident Type')
plt.ylabel('Proportion')
plt.title('Proportions of vehicle types within each Accident Type')
plt.savefig("data/plots/vehicle_prop_by_type.pdf", bbox_inches="tight", pad_inches=0)
plt.show()

Let's also look at the marginal frequencies of all vehicles regardless of their accident types.

In [ ]:
marginal_totals_counts = beteiligte_features_vehicle_type_counts[beteiligte_features_vehicle_type_counts.columns[1:-1]].sum(axis=0)
marginal_frequencies = pd.Series(marginal_totals_counts / marginal_totals_counts.sum())

marginal_frequencies = pd.DataFrame(marginal_frequencies).T
marginal_frequencies.index = ["Overall"]


In [ ]:
vehicle_types = ["Car", "Truck", "Unknown (Hit and run)", "Pedestrian", "Bike", "Other", "Tram", "Bus", "Motorcycle", "E-Scooter"]

ax = marginal_frequencies.plot(kind='bar', stacked=True, colormap='Paired', width=0.1, figsize=(3,6))
ax.legend_.remove()
ax.set_xticks([])
ax.set_xticklabels([])
plt.legend(vehicle_types, title="Vehicle Type", loc="upper left", bbox_to_anchor=(1.05, 1))
plt.title("Overall Distribution of Vehicle Types")
plt.ylabel("Proportion")
plt.savefig("data/vehicle_prop.pdf", bbox_inches="tight", pad_inches=0)
plt.show()


In [ ]:
# for paper: Combine both from above into one plot (sharing legend)
vehicle_types = ["Car", "Truck", "Unknown (Hit and run)", "Pedestrian", "Bike", "Other", "Tram", "Bus", "Motorcycle", "E-Scooter"]

fig, axes = plt.subplots(1, 2, figsize=(10,6), gridspec_kw={'width_ratios': [1, 3]})

ax1 = marginal_frequencies.plot(kind='bar', stacked=True, colormap='Paired', width=0.1, ax=axes[0])
ax1.legend_.remove()
ax1.set_xticks([])
ax1.set_xticklabels([])
ax1.set_ylabel("Proportion")
ax1.set_title("Overall Distribution of Vehicle Types")

ax2 = proportions.plot(kind='bar', stacked=True, colormap="Paired", ax=axes[1])
ax2.legend_.remove()
ax2.set_xticks(range(7))
ax2.set_xticklabels(accident_types, rotation=45, ha='right')
ax2.set_xlabel('Accident Type')
ax2.set_ylabel('Proportion')
ax2.set_title('Proportions of Vehicle Types within Each Accident Type')

# Shared legend
handles, labels = ax2.get_legend_handles_labels()
fig.legend(handles, vehicle_types, title="Vehicle Type", loc="upper left", bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.savefig("data/plots/combined_vehicle_prop.pdf", bbox_inches="tight", pad_inches=0)
plt.show()
